In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("ernie55ernie/improved-cicids2017-and-csecicids2018")

print("Path to dataset files:", path)

100%|██████████| 10.2G/10.2G [02:05<00:00, 87.5MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/ernie55ernie/improved-cicids2017-and-csecicids2018/versions/1


This section performs data preprocessing, including converting IP addresses to numerical values, encoding attack labels into binary format, and normalizing selected numerical features to ensure consistency for model training.

In [ ]:
import shutil
import os

# Path to the extracted dataset
source_path = "/root/.cache/kagglehub/datasets/ernie55ernie/improved-cicids2017-and-csecicids2018/versions/1"
destination_path = "/content/network_intrusion"  # New location in the home directory

# Move the dataset
shutil.move(source_path, destination_path)

# Verify the move
print("Files moved to:", os.path.abspath(destination_path))
print("Files in the new location:", os.listdir(destination_path))


Files moved to: /content/network_intrusion
Files in the new location: ['CICIDS2017_improved', 'CSECICIDS2018_improved']


In [ ]:
destination_path  ="/content/network_intrusion/CICIDS2017_improved"
files = os.listdir(destination_path)
print("Files in the dataset:", files)

Files in the dataset: ['tuesday.csv', 'monday.csv', 'friday.csv', 'wednesday.csv', 'thursday.csv']


In [ ]:
import pandas as pd

# Combine all CSV files into one "DataFrame
combined_data = pd.DataFrame()

for file_name in files:
    if file_name.endswith(".csv"):  # Only process CSV files
        file_path = os.path.join(destination_path, file_name)
        print(f"Loading file: {file_name}")
        temp_data = pd.read_csv(file_path)
        combined_data = pd.concat([combined_data, temp_data], ignore_index=True)

print("Combined data shape:", combined_data.shape)
print(combined_data.head())

Loading file: tuesday.csv
Loading file: monday.csv
Loading file: friday.csv
Loading file: wednesday.csv
Loading file: thursday.csv
Combined data shape: (2099976, 91)
   id                                Flow ID        Src IP  Src Port  \
0   1  192.168.10.5-192.168.10.3-49159-445-6  192.168.10.5     49159   
1   2                  8.6.0.1-8.0.6.4-0-0-0       8.6.0.1         0   
2   3   192.168.10.5-192.168.10.3-123-123-17  192.168.10.5       123   
3   4  192.168.10.3-192.168.10.1-60280-53-17  192.168.10.3     60280   
4   5  192.168.10.3-192.168.10.1-61995-53-17  192.168.10.3     61995   

         Dst IP  Dst Port  Protocol                   Timestamp  \
0  192.168.10.3       445         6  2017-07-04 11:53:44.398274   
1       8.0.6.4         0         0  2017-07-04 11:54:12.355218   
2  192.168.10.3       123        17  2017-07-04 11:54:32.240412   
3  192.168.10.1        53        17  2017-07-04 11:55:07.615878   
4  192.168.10.1        53        17  2017-07-04 11:54:12.427035   

In [ ]:
combined_data.columns

Index(['id', 'Flow ID', 'Src IP', 'Src Port', 'Dst IP', 'Dst Port', 'Protocol',
       'Timestamp', 'Flow Duration', 'Total Fwd Packet', 'Total Bwd packets',
       'Total Length of Fwd Packet', 'Total Length of Bwd Packet',
       'Fwd Packet Length Max', 'Fwd Packet Length Min',
       'Fwd Packet Length Mean', 'Fwd Packet Length Std',
       'Bwd Packet Length Max', 'Bwd Packet Length Min',
       'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Flow Bytes/s',
       'Flow Packets/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max',
       'Flow IAT Min', 'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std',
       'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean',
       'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags',
       'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'Fwd RST Flags',
       'Bwd RST Flags', 'Fwd Header Length', 'Bwd Header Length',
       'Fwd Packets/s', 'Bwd Packets/s', 'Packet Length Min',
       'Packet Length Max', 'Packet Length

In [ ]:
features = ["Src IP", "Src Port", "Dst IP", "Dst Port", "Protocol",
            "Total Fwd Packet", "Packet Length Min", "Packet Length Max", "Packet Length Mean",
            'FIN Flag Count', 'SYN Flag Count',
       'RST Flag Count', 'PSH Flag Count', 'ACK Flag Count', 'URG Flag Count', 'Label']
labels = "Label"


In [ ]:
final_data = combined_data[features]
final_data

,Src IP,Src Port,Dst IP,Dst Port,Protocol,Total Fwd Packet,Packet Length Min,Packet Length Max,Packet Length Mean,FIN Flag Count,SYN Flag Count,RST Flag Count,PSH Flag Count,ACK Flag Count,URG Flag Count,Label
0,192.168.10.5,49159,192.168.10.3,445,6,10,0,1,0.500000,0,0,0,0,20,0,BENIGN
1,8.6.0.1,0,8.0.6.4,0,0,26,0,0,0.000000,0,0,0,0,0,0,BENIGN
2,192.168.10.5,123,192.168.10.3,123,17,4,48,48,48.000000,0,0,0,0,0,0,BENIGN
3,192.168.10.3,60280,192.168.10.1,53,17,1,44,240,142.000000,0,0,0,0,0,0,BENIGN
4,192.168.10.3,61995,192.168.10.1,53,17,1,48,80,64.000000,0,0,0,0,0,0,BENIGN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2099971,192.168.10.8,61232,192.168.10.19,21571,6,2,0,0,0.000000,0,2,2,0,2,0,Infiltration - Portscan
2099972,192.168.10.15,55355,192.168.10.3,53,17,2,38,54,46.000000,0,0,0,0,0,0,BENIGN
2099973,192.168.10.14,63524,192.168.10.3,53,17,2,32,104,68.000000,0,0,0,0,0,0,BENIGN
2099974,192.168.10.15,55298,192.168.10.50,22,6,42,0,976,105.295455,4,4,0,48,86,0,BENIGN


In [ ]:
final_data.to_csv("final_data.csv", index=False)